# 06. アンサンブル: 5つのモデルを、重みを学んで混ぜる

**このノートブックの目的:** 04・05で作った5つのモデルの予測を混ぜ、単体より高いAUCを出す。どのモデルが混ぜたスコアに効いているかも確かめる。

**使うもの:** 5つのモデルのOOF(LightGBM・XGBoostは05で調整したもの、CatBoost・RealMLP・TabPFNはKaggleのGPUで学習したもの)。

**結論:** 2段目にロジスティック回帰を使ったスタックで **OOF AUC 0.961685**(Private LB 0.96208、上位29%)。重く使われたのは、木と仕組みの違うTabPFN(係数0.42)とRealMLP(0.35)だった。

## 準備

- ライブラリと、複数のノートブックで使う共通処理(`src/features.py`: データの読み込みと特徴量の作り方、`src/modeling.py`: 交差検証・ハイパラ・スタッキング)を読み込む。
- train・testを読み込み、目的変数 `satisfaction` を0/1の配列 `y` にする。

In [1]:
import sys

sys.path.insert(0, "../src")  # 共通処理(features.py・modeling.py・public_oof.py)を読み込む
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import features
import modeling

plt.rcParams["font.family"] = "Meiryo"  # グラフの日本語ラベル用
DATA = features.DATA_DIR
train, test = features.load_train_and_test()
y = train[features.TARGET].astype(int).values
print(f"train {len(train):,}行・test {len(test):,}行、満足率 {y.mean():.3f}")

train 699,635行・test 299,844行、満足率 0.444


In [2]:
predictions = {"LightGBM": np.load(DATA / "oof_lightgbm_tuned.npy"), "XGBoost": np.load(DATA / "oof_xgboost_tuned.npy"),
               "CatBoost": np.load(DATA / "oof_catboost_tuned.npy"), "RealMLP": np.load(DATA / "oof_realmlp.npy"),
               "TabPFN": np.load(DATA / "oof_tabpfn.npy")}
names = list(predictions)
single = pd.Series({name: roc_auc_score(y, p) for name, p in predictions.items()}, name="単体のOOF AUC")
correlation = modeling.rank_correlation_table(predictions)
pd.concat([single.round(6), correlation.round(3).add_suffix("との相関")], axis=1)

,単体のOOF AUC,LightGBMとの相関,XGBoostとの相関,CatBoostとの相関,RealMLPとの相関,TabPFNとの相関
LightGBM,0.960853,1.000,0.992,0.985,0.970,0.965
XGBoost,0.960741,0.992,1.000,0.980,0.966,0.962
CatBoost,0.960936,0.985,0.980,1.000,0.971,0.963
RealMLP,0.961192,0.970,0.966,0.971,1.000,0.966
TabPFN,0.960844,0.965,0.962,0.963,0.966,1.000


**表の見方:** 左=単体のOOF AUC、右=モデルどうしの予測の順位相関(1に近いほど似た予測)。単体の強さはほぼ同じ(0.9608〜0.9612)で、似かたが違う。木どうしは0.98〜0.99と高く、TabPFNはほかの4モデルと0.96台と低い。

## 1. 2段目で重みを学んで混ぜる(スタッキング)

**目的:** 単体の強さが近く予測の違う5モデルを混ぜれば、得意・不得意を補い合い、AUCが上がると考えた。

**検証方法:**
- 1段目: 5モデルとも同じ5分割で学習したOOF(各行を、その行を学習していないモデルで予測した値)。
- 2段目: OOFをlogit(log(p/(1−p)))に直し、答えに一番合う重みと切片をロジスティック回帰で学ぶ(重みの合計は1に限らない)。
- 公平に測る: 重みを決めた行で採点すると点数が甘くなるので、「4つで重みを学び、残り1つで採点」を5回くり返して、スタックのOOF AUCを出す。
- 比べる相手: 予測を順位に直し、重み(合計1、0.1刻み)を学習側の行で選んで平均する方法(RealMLPとLightGBMを混ぜたときに使っていた方法)。

In [3]:
oof_matrix = np.column_stack([predictions[name] for name in names])
logits = modeling.to_logit(oof_matrix)
stack_oof, coefficients = modeling.cross_validate_logistic_stack(logits, y)
rank_oof = modeling.cross_validate_rank_blend(oof_matrix, y, weight_step=0.1)  # 重みの組み合わせは0.1刻み(1,001通り)
print(f"ロジスティック回帰のスタック OOF AUC {roc_auc_score(y, stack_oof):.6f}")
print(f"順位の重み付き平均       OOF AUC {roc_auc_score(y, rank_oof):.6f}")
print(f"差 {roc_auc_score(y, stack_oof) - roc_auc_score(y, rank_oof):+.6f}")
pd.Series(coefficients.mean(axis=0), index=names, name="2段目の係数(5回の平均)").round(3)

ロジスティック回帰のスタック OOF AUC 0.961679


順位の重み付き平均       OOF AUC 0.961642


差 +0.000037


LightGBM    0.047
XGBoost     0.053
CatBoost    0.129
RealMLP     0.352
TabPFN      0.417
Name: 2段目の係数(5回の平均), dtype: float64

#### 結果と考察

- ロジスティック回帰のスタックの方が、順位の重み付き平均より高かった(記録では、重みを0.05刻みで選んだ平均より+0.000039)。正則化により、モデルの本数が増えても重みが一部のモデルに偏りすぎるのを抑えられる利点もあるので、2段目はロジスティック回帰にした。調べた上位の公開ノートブック7本も、2段目をすべてlogitのロジスティック回帰にしていた。
- 重く使われたのはTabPFNとRealMLPだった。木との順位相関が低く、木が間違える行を補えたと考えられる(推測。行ごとには確かめていない)。

## 2. 1モデルずつ抜いて、スタックのOOFを比べる

**目的:** 係数の大きさだけでは、そのモデルが無いと困るかは分からない(似たモデルが代わりを務めることがある)。抜いたときの変化で役割を確かめる。

In [4]:
full = roc_auc_score(y, stack_oof)
drop_one = {}
for j, name in enumerate(names):
    rest = [k for k in range(len(names)) if k != j]
    drop_one[name] = roc_auc_score(y, modeling.cross_validate_logistic_stack(logits[:, rest], y)[0]) - full
pd.Series(drop_one, name="抜いたときのOOF AUCの変化").round(6)

LightGBM    0.000000
XGBoost    -0.000006
CatBoost   -0.000025
RealMLP    -0.000199
TabPFN     -0.000142
Name: 抜いたときのOOF AUCの変化, dtype: float64

#### 結果と考察

- 係数の大きいRealMLPとTabPFNは、抜くと大きく下がった(記録では−0.000188・−0.000141)。
- LightGBMは抜いても変わらなかった。XGBoost・CatBoostと予測が似ていて(相関0.98〜0.99)、役割が重なっていた。

## Summary

| 決めたこと | 値 |
|---|---|
| 2段目 | logitのロジスティック回帰(4つで学び残り1つで採点、で評価) |
| 5モデルのスタック | OOF AUC 0.961685 / Private LB 0.96208(上位29%) |
| 効いたモデル | RealMLP・TabPFN(木と仕組みが違う) |

上位との差は、混ぜるモデルの数と種類から来ていると考え、07で他の参加者が公開したOOFを加える。